# 🚀 IEEE-CIS Fraud Detection - XGBoost on Google Colab
This notebook provides an end-to-end, memory-optimized machine learning pipeline using **XGBoost** with GPU acceleration on Google Colab to detect fraudulent transactions.

---

### 📋 Overview & Dataset Info
- **Dataset**: IEEE-CIS Fraud Detection
- **Files**:
  - `train_transaction.csv` & `train_identity.csv`
  - `test_transaction.csv` & `test_identity.csv`
  - `sample_submission.csv`
- **Target Variable**: `isFraud` (Binary classification: 0 = Legitimate, 1 = Fraud)
- **Primary Metric**: ROC-AUC Score

---

### ⚙️ Google Colab Hardware Setup
Ensure you enable GPU acceleration for faster training:
1. Go to **Runtime** > **Change runtime type**
2. Under **Hardware accelerator**, select **GPU** (T4 GPU recommended)
3. Click **Save**

## 1. Environment Setup & GPU Verification
Import essential packages (`xgboost`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`) and check GPU availability.

In [ ]:
# Verify GPU availability in Colab
!nvidia-smi

import sys
import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"XGBoost Version: {xgb.__version__}")

## 2. Load Dataset in Google Colab
Choose **Option A** (Google Drive) or **Option B** (Direct Upload / Kaggle API).

In [ ]:
# Option A: Load from Google Drive (Recommended)
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where your dataset is stored in Google Drive:
DATA_DIR = '/content/drive/MyDrive/ieee-cis-fraud-detection' 

# Alternative Option B: If files are uploaded directly to Colab root folder:
# DATA_DIR = '.'

import os
if not os.path.exists(DATA_DIR):
    print(f"Directory '{DATA_DIR}' not found! Defaulting to current working directory '.'")
    DATA_DIR = '.'

print(f"Loading data from: {DATA_DIR}")
train_trans_path = os.path.join(DATA_DIR, 'train_transaction.csv')
train_id_path = os.path.join(DATA_DIR, 'train_identity.csv')
test_trans_path = os.path.join(DATA_DIR, 'test_transaction.csv')
test_id_path = os.path.join(DATA_DIR, 'test_identity.csv')
sub_path = os.path.join(DATA_DIR, 'sample_submission.csv')

## 3. Data Downcasting & Merging
The IEEE-CIS dataset contains over 400 columns across transactions and identities. To avoid Colab RAM out-of-memory (OOM) errors, we use an automatic memory downcasting function `reduce_mem_usage`.

In [ ]:
def reduce_mem_usage(df, verbose=True):
    """Iterate through all columns of a dataframe and downcast numeric data types to reduce memory."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and not pd.api.types.is_categorical_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32) # float32 is safer than float16 for XGBoost
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage decreased to {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

print("--- Loading Train Datasets ---")
train_trans = pd.read_csv(train_trans_path)
train_trans = reduce_mem_usage(train_trans)

train_id = pd.read_csv(train_id_path)
train_id = reduce_mem_usage(train_id)

print("\n--- Loading Test Datasets ---")
test_trans = pd.read_csv(test_trans_path)
test_trans = reduce_mem_usage(test_trans)

test_id = pd.read_csv(test_id_path)
# Note: test_identity.csv columns use hyphens (id-01) vs underscores (id_01) in train_identity.csv
test_id.columns = [col.replace('-', '_') for col in test_id.columns]
test_id = reduce_mem_usage(test_id)

print("\n--- Merging Datasets ---")
train_df = pd.merge(train_trans, train_id, on='TransactionID', how='left')
test_df = pd.merge(test_trans, test_id, on='TransactionID', how='left')

del train_trans, train_id, test_trans, test_id
gc.collect()

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape:  {test_df.shape}")

## 4. Exploratory Data Analysis (EDA)
Inspect the class balance of `isFraud`, transaction amount distributions, and missing value metrics.

In [ ]:
# Target distribution plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='isFraud', data=train_df, ax=ax[0], palette=['#2ecc71', '#e74c3c'])
ax[0].set_title('Target Class Counts (0: Legitimate, 1: Fraud)', fontsize=14)
ax[0].set_ylabel('Count')

fraud_counts = train_df['isFraud'].value_counts(normalize=True) * 100
ax[1].pie(fraud_counts, labels=['Legitimate (0)', 'Fraud (1)'], autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.1))
ax[1].set_title('Target Class Percentage', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Total Transactions: {len(train_df)}")
print(f"Fraudulent Transactions: {train_df['isFraud'].sum()} ({fraud_counts[1]:.2f}%)")

In [ ]:
# Transaction Amount Distribution
plt.figure(figsize=(12, 5))
sns.kdeplot(train_df[train_df['isFraud'] == 0]['TransactionAmt'], label='Legitimate', color='green', shade=True, clip=(0, 1000))
sns.kdeplot(train_df[train_df['isFraud'] == 1]['TransactionAmt'], label='Fraud', color='red', shade=True, clip=(0, 1000))
plt.title('Transaction Amount Distribution ($0 - $1000)', fontsize=14)
plt.xlabel('Transaction Amount ($)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 5. Feature Engineering & Categorical Encoding
1. **Time Features**: Extract hour and day features from `TransactionDT`.
2. **Log Transformation**: Log transform `TransactionAmt` to reduce skewness.
3. **Categorical Encoding**: Label encode all categorical object columns so XGBoost can ingest them efficiently.

In [ ]:
# Time features from TransactionDT
def create_time_features(df):
    df['DT_M'] = (df['TransactionDT'] / (3600 * 24 * 30)).astype(np.int8)
    df['DT_W'] = (df['TransactionDT'] / (3600 * 24 * 7)).astype(np.int8)
    df['DT_D'] = (df['TransactionDT'] / (3600 * 24)).astype(np.int16)
    df['DT_hour'] = ((df['TransactionDT'] / 3600) % 24).astype(np.int8)
    df['DT_day_of_week'] = ((df['TransactionDT'] / (3600 * 24)) % 7).astype(np.int8)
    return df

print("Extracting time features...")
train_df = create_time_features(train_df)
test_df = create_time_features(test_df)

# Log transform TransactionAmt
train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
test_df['TransactionAmt_log'] = np.log1p(test_df['TransactionAmt'])

# Identify categorical columns
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
            'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
            'DeviceType', 'DeviceInfo'] + [f'id_{i:02d}' for i in range(12, 39)]

cat_cols = [col for col in cat_cols if col in train_df.columns]

print(f"Encoding {len(cat_cols)} categorical features...")
for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)
    
    le = LabelEncoder()
    le.fit(list(train_df[col].values) + list(test_df[col].values))
    train_df[col] = le.transform(train_df[col].values)
    test_df[col] = le.transform(test_df[col].values)

train_df = reduce_mem_usage(train_df)
test_df = reduce_mem_usage(test_df)
gc.collect()
print("Feature Engineering & Encoding Complete!")

## 6. Model Training with GPU-Accelerated XGBoost
We prepare feature sets `X` and target `y`, and train an **XGBoost Classifier** using 5-Fold Stratified K-Fold Cross-Validation.

In [ ]:
# Define features to drop (TransactionID, target, and raw TransactionDT)
features_to_drop = ['TransactionID', 'isFraud', 'TransactionDT']
features = [col for col in train_df.columns if col not in features_to_drop]

X = train_df[features]
y = train_df['isFraud']
X_test = test_df[features]

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")
print(f"Test feature shape:   {X_test.shape}")

In [ ]:
# Set up 5-Fold Stratified K-Fold CV
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
feature_importance_df = pd.DataFrame()

# Check XGBoost device configuration
try:
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'learning_rate': 0.05,
        'max_depth': 9,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'missing': -999,
        'tree_method': 'hist',
        'device': 'cuda',  # Uses Colab GPU
        'random_state': 42
    }
    # Test dummy fit to verify GPU
    dummy_model = xgb.XGBClassifier(**xgb_params, n_estimators=2)
    dummy_model.fit(X.iloc[:100], y.iloc[:100])
    print("XGBoost GPU acceleration (cuda) is ACTIVE!")
except Exception as e:
    print(f"GPU device setup notice: {e}")
    print("Falling back to standard CPU tree_method='hist'")
    xgb_params['device'] = 'cpu'

fold_auc_scores = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n================ Fold {fold + 1} / {N_SPLITS} ================")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        **xgb_params,
        n_estimators=1000,
        early_stopping_rounds=50
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100
    )
    
    # Predict validation set
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_auc_scores.append(fold_auc)
    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")
    
    # Predict test set (accumulate fold predictions)
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    # Track feature importances
    fold_importance = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importances_
    })
    feature_importance_df = pd.concat([feature_importance_df, fold_importance], axis=0)

overall_auc = roc_auc_score(y, oof_preds)
elapsed = time.time() - start_time
print(f"\n================ Training Finished in {elapsed/60:.2f} mins ================")
print(f"Mean Fold ROC-AUC: {np.mean(fold_auc_scores):.5f} +/- {np.std(fold_auc_scores):.5f}")
print(f"Overall OOF ROC-AUC Score: {overall_auc:.5f}")

## 7. Model Diagnostics & Visualizations
Evaluate the out-of-fold predictions, plot the ROC Curve, Confusion Matrix, and top feature importances.

In [ ]:
# 1. ROC Curve
fpr, tpr, thresholds = roc_curve(y, oof_preds)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'XGBoost OOF ROC-AUC = {overall_auc:.5f}', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('Out-Of-Fold ROC Curve', fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(True)
plt.show()

In [ ]:
# 2. Confusion Matrix & Classification Report
optimal_threshold = 0.5
oof_binary = (oof_preds >= optimal_threshold).astype(int)

cm = confusion_matrix(y, oof_binary)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix (Threshold = {optimal_threshold})', fontsize=14)
plt.show()

print("\n--- Classification Report ---")
print(classification_report(y, oof_binary, target_names=['Legitimate (0)', 'Fraud (1)']))

In [ ]:
# 3. Top 25 Feature Importances
mean_importance = feature_importance_df.groupby('feature')['importance'].mean().reset_index()
top_features = mean_importance.sort_values(by='importance', ascending=False).head(25)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=top_features, palette='viridis')
plt.title('Top 25 Most Important Features in XGBoost Model', fontsize=14)
plt.xlabel('Mean Feature Importance (Gain)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Generate & Download Submission File
Format predictions according to `sample_submission.csv` and export to `submission.csv`.

In [ ]:
sub = pd.DataFrame({
    'TransactionID': test_df['TransactionID'],
    'isFraud': test_preds
})

sub_filename = 'submission.csv'
sub.to_csv(sub_filename, index=False)
print(f"Saved submission file: {sub_filename}")

# Display submission overview & sanity check
print("\n--- Submission Sanity Check ---")
print(f"Rows count: {len(sub)}")
print(f"Missing values: {sub['isFraud'].isnull().sum()}")
print(f"Predictions min: {sub['isFraud'].min():.4f}, max: {sub['isFraud'].max():.4f}, mean: {sub['isFraud'].mean():.4f}")
print("\nFirst 10 rows:")
print(sub.head(10))

# Automatically trigger download in Colab
try:
    from google.colab import files
    files.download(sub_filename)
except Exception as e:
    print(f"To download locally, find '{sub_filename}' in the Colab file browser sidebar.")